# Pull all of 2025

*Data Analysis in Python · Topic 12.3*

Your media team wants all of 2025's daily numbers by channel. That's 33 pages, more than the API answers in 10 seconds. Write a function that waits and tries again after a 429 or a 503, and page through the year with it.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell. It reads the course API key from Colab Secrets, as in topic 12.1.
2. In the first cell marked ✏️, write the function `get_json(url, params, tries=5)`. It sends a GET request with `HEADERS` and `params`, up to `tries` times with `for attempt in range(tries):`, and:
   - after a 429 or a 503, waits the seconds in the `Retry-After` header and tries again
   - after any other error, stops with `raise_for_status()`
   - returns the answer's JSON when the request works
   - after `tries` failed tries, stops with `raise RuntimeError(...)`
3. In the second cell marked ✏️, page through the `reports/daily` endpoint for `START_DATE` to `END_DATE`, with `channel` as the dimension and `PAGE_SIZE` rows per page, using `get_json()` for every request. Create the `year_2025` table from the rows, and `row_count`: the `row_count` the API reported.
4. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output. The check calls your function once with a simulated 503, so it takes a few seconds.

**Expected output**: `get_json()` gives up with a `RuntimeError` on a simulated 503. The `year_2025` table has 3,285 rows, the same as `row_count`, with 1,431,715 sessions and \$3,408,186.35 of revenue.

## Parameters

In [ ]:
import time

import pandas as pd
import requests
from google.colab import userdata

BASE_URL = "https://api.teamsimmer.com/v1/"
API_KEY = userdata.get("SIMMER_API_KEY")  # Colab Secrets, as in topic 12.1
HEADERS = {"X-API-Key": API_KEY}
START_DATE, END_DATE = "2025-01-01", "2025-12-31"
PAGE_SIZE = 100

## The retry function

In [ ]:
# ✏️ Write get_json()


## The year

In [ ]:
# ✏️ Create year_2025 and row_count


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell reads your variables and the cells that have run, and calls get_json() once with a simulated error.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    get_json = found.get("get_json")
    if not callable(get_json):
        results.append((False, "get_json should be a function: def get_json(url, params, tries=5): …"))
    else:
        try:
            outcome = get_json(found["BASE_URL"] + "reports/daily", {"simulate_error": "503"}, tries=2)
        except RuntimeError:
            outcome = "gave up"
        except Exception as error:
            outcome = error
        if outcome == "gave up":
            results.append((True, "get_json() tries again after a 503, and gives up with a RuntimeError when it keeps failing."))
        elif isinstance(outcome, Exception) and "503" in str(outcome):
            results.append((False, "get_json() stopped at the first 503. After a 429 or a 503, wait the Retry-After seconds and try again, and raise RuntimeError after tries failed tries."))
        elif isinstance(outcome, Exception):
            results.append((False, f"get_json() stopped with {type(outcome).__name__}: {outcome}. After tries failed tries, it should raise RuntimeError."))
        else:
            results.append((False, "get_json() returned something for a request that always fails with 503. It should try again, then raise RuntimeError after tries failed tries."))

    year, row_count = found.get("year_2025"), found.get("row_count")
    if not isinstance(year, pd.DataFrame) or "sessions" not in year.columns:
        results.append((False, "year_2025 should be a DataFrame of every page's rows."))
    elif row_count != 3285:
        results.append((False, f"row_count should be the row_count the API reported: body[\"row_count\"], 3,285. It is {row_count!r}."))
    elif len(year) == 100:
        results.append((False, "The year_2025 table has only the first page: 100 rows. Page through every page, as in topic 12.2."))
    elif len(year) != row_count:
        results.append((False, f"The year_2025 table has {len(year):,} rows, and the API reported {row_count:,}. Page through every page."))
    elif int(year["sessions"].sum()) != 1431715 or abs(year["revenue"].sum() - 3408186.35) > 0.01:
        results.append((False, f"The year_2025 table should have 1,431,715 sessions and $3,408,186.35 of revenue, for all of 2025. It has {year['sessions'].sum():,} and ${year['revenue'].sum():,.2f}."))
    else:
        results.append((True, "3,285 rows from 33 pages, as the API reported: 1,431,715 sessions and $3,408,186.35 of revenue."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
RETRY_STATUSES = {429, 503}


def get_json(url, params, tries=5):
    """Return the JSON body of a GET request, waiting and trying again after a 429 or 503."""
    for attempt in range(tries):
        response = requests.get(url, headers=HEADERS, params=params)
        if response.status_code not in RETRY_STATUSES:
            response.raise_for_status()
            return response.json()
        wait = int(response.headers.get("Retry-After", 1))
        print(f"{response.status_code}, waiting {wait} s")
        time.sleep(wait)
    raise RuntimeError(f"Still {response.status_code} after {tries} tries: {url}")
```

```python
params = {"start_date": START_DATE, "end_date": END_DATE, "dimensions": "channel", "page_size": PAGE_SIZE}
rows = []
while True:
    body = get_json(BASE_URL + "reports/daily", params)
    rows.extend(body["rows"])
    if "next_page_token" not in body:
        break
    params["page_token"] = body["next_page_token"]
year_2025 = pd.DataFrame(rows)
row_count = body["row_count"]
len(year_2025), row_count
```

</details>